# 08 — Veri Birleştirme ve Yeni Split
labels_v2.csv'den Drama/Comedy downsample et, poster kontrolu yap,
iterative-stratification ile yeni 70/15/15 split olustur.

In [ ]:
import ast
import pickle
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer

# --- Yerel ---
CODE_ROOT = Path('../')
DATA_ROOT = Path('../')

# --- Colab ---
# from google.colab import drive
# drive.mount('/content/drive')
# CODE_ROOT = Path('/content/drive/MyDrive/film-genre-project')
# DATA_ROOT = Path('/content/drive/MyDrive/film-genre-project-data')

LABELS_V2   = DATA_ROOT / 'labels_v2.csv'
POSTERS_DIR = DATA_ROOT / 'posters'
TRAIN_CSV   = DATA_ROOT / 'train.csv'
VAL_CSV     = DATA_ROOT / 'val.csv'
TEST_CSV    = DATA_ROOT / 'test.csv'
MLB_PKL     = DATA_ROOT / 'mlb.pkl'

SEED           = 42
DOWNSAMPLE_TO  = 2500  # Drama ve Comedy

TARGET_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Family', 'Fantasy', 'History',
    'Horror', 'Mystery', 'Romance', 'Science Fiction', 'Thriller'
]

print('labels_v2.csv:', LABELS_V2.exists())

## 1. Veri Yukleme ve Temizleme

In [ ]:
df = pd.read_csv(LABELS_V2, dtype={'tmdb_id': str})

# genres parse — pipeline '|' ile, preprocessing ast ile yazabilir
def parse_genres(val):
    if isinstance(val, str) and val.startswith('['):
        return ast.literal_eval(val)
    if isinstance(val, str):
        return [g.strip() for g in val.split('|') if g.strip()]
    return []

df['genres'] = df['genres'].apply(parse_genres)
df['genres'] = df['genres'].apply(lambda gs: [g for g in gs if g in TARGET_GENRES])

# Poster kontrolu
df['has_poster'] = df['tmdb_id'].apply(lambda t: (POSTERS_DIR / f'{t}.jpg').exists())
no_poster = (~df['has_poster']).sum()
df = df[df['has_poster'] & (df['genres'].map(len) > 0)].reset_index(drop=True)

# Duplikat tmdb_id kaldir
before_dedup = len(df)
df = df.drop_duplicates(subset='tmdb_id').reset_index(drop=True)

print(f'Ham kayit      : {len(pd.read_csv(LABELS_V2)):,}')
print(f'Postersiz      : {no_poster}')
print(f'Duplikat       : {before_dedup - len(df)}')
print(f'Temiz film     : {len(df):,}')
print(f'Ort. tur/film  : {df["genres"].map(len).mean():.2f}')

## 2. Drama ve Comedy Downsample

In [ ]:
rng = np.random.default_rng(SEED)

def downsample_genre(dataframe, genre, target, random_gen):
    has_genre = dataframe['genres'].apply(lambda gs: genre in gs)
    genre_films = dataframe[has_genre]
    if len(genre_films) <= target:
        return dataframe
    drop_n = len(genre_films) - target
    drop_idx = random_gen.choice(genre_films.index.values, size=drop_n, replace=False)
    return dataframe.drop(index=drop_idx).reset_index(drop=True)

print(f'Drama oncesi  : {df["genres"].apply(lambda gs: "Drama" in gs).sum():,}')
df = downsample_genre(df, 'Drama', DOWNSAMPLE_TO, rng)
print(f'Drama sonrasi : {df["genres"].apply(lambda gs: "Drama" in gs).sum():,}')

print(f'Comedy oncesi : {df["genres"].apply(lambda gs: "Comedy" in gs).sum():,}')
df = downsample_genre(df, 'Comedy', DOWNSAMPLE_TO, rng)
print(f'Comedy sonrasi: {df["genres"].apply(lambda gs: "Comedy" in gs).sum():,}')

print(f'\nToplam film (downsample sonrasi): {len(df):,}')

## 3. Son Tur Dagilimi

In [ ]:
genre_counts = Counter(g for gs in df['genres'] for g in gs)

genres_sorted = sorted(TARGET_GENRES, key=lambda g: genre_counts.get(g, 0))
counts_sorted = [genre_counts.get(g, 0) for g in genres_sorted]

print('Son dagilim:')
for g, c in zip(genres_sorted, counts_sorted):
    bar = '#' * (c // 50)
    print(f'  {g:20s}: {c:5,}  {bar}')

max_c = max(counts_sorted)
min_c = min(counts_sorted)
print(f'\nMax/Min orani: {max_c/min_c:.2f}x (hedef <= 1.5x)')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(genres_sorted, counts_sorted, color='steelblue', alpha=0.85)
ax.axvline(2000, color='navy', linestyle='--', linewidth=1.5, label='Min hedef: 2,000')
ax.set_xlabel('Film Sayisi')
ax.set_title('Dengelenmis Veri Seti — Tur Dagilimi')
ax.legend()
plt.tight_layout()
plt.show()

## 4. MultiLabelBinarizer

In [ ]:
mlb = MultiLabelBinarizer(classes=TARGET_GENRES)
Y = mlb.fit_transform(df['genres'])

print(f'Y shape: {Y.shape}  (film x tur)')
print(f'Siniflar: {mlb.classes_.tolist()}')

with open(MLB_PKL, 'wb') as f:
    pickle.dump(mlb, f)
print(f'mlb.pkl guncellendi: {MLB_PKL}')

## 5. Stratified Split (70/15/15)

In [ ]:
# Once 70 / 30 ayir, sonra 30'u 15/15 yap
splitter1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(splitter1.split(df, Y))

splitter2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(splitter2.split(df.iloc[temp_idx], Y[temp_idx]))
val_idx  = temp_idx[val_idx]
test_idx = temp_idx[test_idx]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

print(f'Train : {len(train_df):,}')
print(f'Val   : {len(val_df):,}')
print(f'Test  : {len(test_df):,}')
print(f'Toplam: {len(train_df)+len(val_df)+len(test_df):,}')

In [ ]:
# Splitler arasi dagilim dengeli mi?
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
splits = [('Train', train_df), ('Val', val_df), ('Test', test_df)]

for ax, (name, split_df) in zip(axes, splits):
    cnt = Counter(g for gs in split_df['genres'] for g in gs)
    vals = [cnt.get(g, 0) for g in TARGET_GENRES]
    ax.barh(TARGET_GENRES, vals, color='steelblue', alpha=0.8)
    ax.set_title(f'{name} ({len(split_df):,})')
    ax.set_xlabel('Film Sayisi')

plt.suptitle('Split Bazi Tur Dagilimi')
plt.tight_layout()
plt.show()

## 6. CSV Kaydet

In [ ]:
# genres'i string listesi olarak kaydet (preprocessing notebook ile uyumlu)
for split_df, path in [(train_df, TRAIN_CSV), (val_df, VAL_CSV), (test_df, TEST_CSV)]:
    split_df['genres'] = split_df['genres'].apply(str)
    split_df[['tmdb_id', 'title', 'genres']].to_csv(path, index=False)
    print(f'{path.name} yazildi: {len(split_df):,} film')

print('\nPhase 9 icin hazir.')